# 01 · Tumbabiro — Base física híbrida 2050

**Objetivo.** Generar de forma reproducible `physical_2050_base.csv`, separado del entrenamiento neuronal.

La base combina dos ramas:

\[
\text{geometría solar astronómica 2050}
+
\text{climatología clear-sky histórica 1998–2024}
\]

La geometría se calcula con `pvlib` para Tumbabiro; las componentes `Clearsky GHI/DNI/DHI` se construyen mediante mediana histórica por `(día del año, slot de 30 min)` y suavizado circular de 31 días.

**Salida contractual del notebook**

`physical_2050_base.csv` con 17 520 filas y las columnas:

`timestamp`, `Solar Zenith Angle`, `Clearsky GHI`, `Clearsky DNI`, `Clearsky DHI`, `cosz`, `cosz_plus`.

> Este notebook no entrena el PI-LSTM y no usa información del modelo neuronal.

In [ ]:
# Si el entorno no dispone de pvlib:
# %pip install -q pvlib

import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import pvlib

@dataclass(frozen=True)
class CFG:
    data_path: str = "consol_tumbabiro_2026_total.csv"
    output_path: str = "physical_2050_base.csv"
    metadata_path: str = "physical_2050_metadata.json"

    target_year: int = 2050
    freq_minutes: int = 30
    doy_window: int = 15

    latitude: float = 0.45
    longitude: float = -78.8
    altitude: float = 2050.0
    timezone: str = "America/Guayaquil"

cfg = CFG()
cfg

In [ ]:
# Nombres canónicos
COL_DATE  = "Date"
COL_HOUR  = "Hour"
COL_MIN   = "Minute"
COL_THZ   = "Solar Zenith Angle"
COL_CSGHI = "Clearsky GHI"
COL_CSDNI = "Clearsky DNI"
COL_CSDHI = "Clearsky DHI"

required_raw = [
    COL_DATE, COL_HOUR, COL_MIN,
    COL_CSGHI, COL_CSDNI, COL_CSDHI
]

In [ ]:
# Carga y validación del histórico
df_raw = pd.read_csv(cfg.data_path)

missing = [c for c in required_raw if c not in df_raw.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

d = pd.to_datetime(df_raw[COL_DATE], dayfirst=True, errors="coerce")
ts = (
    d
    + pd.to_timedelta(pd.to_numeric(df_raw[COL_HOUR], errors="coerce"), unit="h")
    + pd.to_timedelta(pd.to_numeric(df_raw[COL_MIN], errors="coerce"), unit="m")
)

df = df_raw.copy()
df["timestamp"] = ts
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

if df["timestamp"].duplicated().any():
    raise ValueError("Se detectaron timestamps duplicados en el histórico.")

# El archivo original es continuo a 30 min; se verifica en vez de asumirlo.
dt = df["timestamp"].diff().dt.total_seconds().dropna() / 60
if not np.allclose(dt.to_numpy(), cfg.freq_minutes):
    raise ValueError("El histórico no es completamente continuo a 30 minutos.")

for c in [COL_CSGHI, COL_CSDNI, COL_CSDHI]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if df[[COL_CSGHI, COL_CSDNI, COL_CSDHI]].isna().any().any():
    raise ValueError("Hay NaN en las variables clear-sky históricas.")

print("Shape histórico:", df.shape)
print("Rango:", df["timestamp"].min(), "->", df["timestamp"].max())

## Climatología clear-sky compatible con el espacio \(k\)

Para cada slot de 30 min y día del año:

1. mediana histórica de `Clearsky GHI/DNI/DHI`;
2. interpolación de eventuales huecos;
3. mediana móvil circular de \(2\times15+1=31\) días.

Se usa el registro histórico completo 1998–2024 porque esta etapa construye una **referencia física-climatológica exógena**; no ajusta los pesos del modelo ni los scalers.

In [ ]:
def build_slot_doy_calendar(target_year: int, freq_minutes: int = 30) -> pd.DataFrame:
    ts = pd.date_range(
        start=f"{target_year}-01-01 00:00:00",
        end=f"{target_year}-12-31 23:30:00",
        freq=f"{freq_minutes}min"
    )
    out = pd.DataFrame({"timestamp": ts})
    out["doy"] = out["timestamp"].dt.dayofyear.astype(np.int16)
    out["slot"] = (
        out["timestamp"].dt.hour * (60 // freq_minutes)
        + out["timestamp"].dt.minute // freq_minutes
    ).astype(np.int16)
    return out

def circular_smooth_by_doy(
    clim_df: pd.DataFrame,
    value_col: str,
    doy_col: str = "doy",
    slot_col: str = "slot",
    window: int = 15
) -> pd.DataFrame:
    pieces = []
    for slot_value, sub in clim_df.groupby(slot_col):
        sub = sub.sort_values(doy_col).reset_index(drop=True).copy()
        full = pd.DataFrame({doy_col: np.arange(1, 366, dtype=int)})
        sub = full.merge(sub[[doy_col, value_col]], on=doy_col, how="left")

        sub[value_col] = sub[value_col].interpolate(limit_direction="both")
        sub[value_col] = sub[value_col].fillna(sub[value_col].median())

        vals = sub[value_col].to_numpy()
        pad = window
        vals_ext = np.concatenate([vals[-pad:], vals, vals[:pad]])
        smoothed = (
            pd.Series(vals_ext)
            .rolling(window=2 * window + 1, center=True, min_periods=1)
            .median()
            .to_numpy()
        )[pad:-pad]

        pieces.append(pd.DataFrame({
            slot_col: slot_value,
            doy_col: np.arange(1, 366, dtype=int),
            value_col: smoothed
        }))
    return pd.concat(pieces, ignore_index=True)

def build_clear_sky_2050_from_historical(
    df_hist_source: pd.DataFrame,
    target_year: int,
    freq_minutes: int = 30,
    doy_window: int = 15
) -> pd.DataFrame:
    src = df_hist_source[["timestamp", COL_CSGHI, COL_CSDNI, COL_CSDHI]].copy()
    src["doy"] = src["timestamp"].dt.dayofyear.astype(np.int16)
    src["slot"] = (
        src["timestamp"].dt.hour * (60 // freq_minutes)
        + src["timestamp"].dt.minute // freq_minutes
    ).astype(np.int16)

    cs_cols = [COL_CSGHI, COL_CSDNI, COL_CSDHI]
    clim_raw = (
        src.groupby(["doy", "slot"])[cs_cols]
        .median()
        .reset_index()
        .sort_values(["slot", "doy"])
    )

    smooth_parts = []
    for col in cs_cols:
        sm = circular_smooth_by_doy(
            clim_raw[["doy", "slot", col]].copy(),
            value_col=col,
            window=doy_window
        )
        smooth_parts.append(sm.rename(columns={col: f"{col}_sm"}))

    clim = smooth_parts[0]
    for sm in smooth_parts[1:]:
        clim = clim.merge(sm, on=["doy", "slot"], how="inner")

    clim = clim.rename(columns={
        f"{COL_CSGHI}_sm": COL_CSGHI,
        f"{COL_CSDNI}_sm": COL_CSDNI,
        f"{COL_CSDHI}_sm": COL_CSDHI,
    })

    cal = build_slot_doy_calendar(target_year, freq_minutes)
    out = cal.merge(clim, on=["doy", "slot"], how="left", validate="many_to_one")

    if out[cs_cols].isna().any().any():
        raise ValueError("NaN al construir clear-sky 2050.")

    for c in cs_cols:
        out[c] = out[c].astype(np.float32)

    return out[["timestamp", *cs_cols]].copy()

## Geometría solar 2050

Se calcula con `pvlib.Location.get_solarposition()` para:

- latitud: \(0.45^\circ\)
- longitud: \(-78.8^\circ\)
- altitud: 2050 m
- zona horaria: `America/Guayaquil`
- resolución: 30 min.

In [ ]:
def build_zenith_2050_from_latlon(
    latitude: float,
    longitude: float,
    altitude: float,
    timezone: str,
    target_year: int,
    freq_minutes: int
) -> pd.DataFrame:
    times_local = pd.date_range(
        start=f"{target_year}-01-01 00:00:00",
        end=f"{target_year}-12-31 23:30:00",
        freq=f"{freq_minutes}min",
        tz=timezone
    )

    site = pvlib.location.Location(
        latitude=latitude,
        longitude=longitude,
        tz=timezone,
        altitude=altitude
    )
    solpos = site.get_solarposition(times_local)

    return pd.DataFrame({
        "timestamp": times_local.tz_localize(None),
        COL_THZ: pd.to_numeric(solpos["zenith"], errors="coerce").to_numpy(dtype=np.float32)
    })

zen_2050 = build_zenith_2050_from_latlon(
    cfg.latitude, cfg.longitude, cfg.altitude, cfg.timezone,
    cfg.target_year, cfg.freq_minutes
)

cs_2050 = build_clear_sky_2050_from_historical(
    df_hist_source=df,
    target_year=cfg.target_year,
    freq_minutes=cfg.freq_minutes,
    doy_window=cfg.doy_window
)

physical_2050 = zen_2050.merge(
    cs_2050, on="timestamp", how="inner", validate="one_to_one"
)

theta = np.deg2rad(physical_2050[COL_THZ].to_numpy(dtype=np.float32))
physical_2050["cosz"] = np.cos(theta).astype(np.float32)
physical_2050["cosz_plus"] = np.maximum(physical_2050["cosz"], 0.0).astype(np.float32)

physical_2050.head()

In [ ]:
# Validaciones estructurales y físicas
expected_rows = 365 * 48
if len(physical_2050) != expected_rows:
    raise ValueError(f"Se esperaban {expected_rows} filas; hay {len(physical_2050)}.")

if physical_2050["timestamp"].duplicated().any():
    raise ValueError("Hay timestamps duplicados.")

dt = physical_2050["timestamp"].diff().dt.total_seconds().dropna() / 60
if not np.allclose(dt.to_numpy(), cfg.freq_minutes):
    raise ValueError("Frecuencia temporal incorrecta.")

critical = [COL_THZ, COL_CSGHI, COL_CSDNI, COL_CSDHI, "cosz", "cosz_plus"]
if physical_2050[critical].isna().any().any():
    raise ValueError("Hay NaN en columnas críticas.")

if not physical_2050[COL_THZ].between(0, 180).all():
    raise ValueError("Zenit fuera de [0,180].")

for c in [COL_CSGHI, COL_CSDNI, COL_CSDHI]:
    if (physical_2050[c] < 0).any():
        raise ValueError(f"Valores negativos en {c}")

night = physical_2050["cosz_plus"] <= 0
night_leak = (
    physical_2050.loc[night, [COL_CSGHI, COL_CSDNI, COL_CSDHI]].abs().to_numpy() > 1e-6
).sum()

closure = (
    physical_2050[COL_CSGHI].to_numpy()
    - (
        physical_2050[COL_CSDNI].to_numpy() * physical_2050["cosz_plus"].to_numpy()
        + physical_2050[COL_CSDHI].to_numpy()
    )
)

print("Night clear-sky nonzero count:", int(night_leak))
print("Clear-sky closure MAE [W/m²]:", float(np.mean(np.abs(closure))))
print("Clear-sky closure RMSE [W/m²]:", float(np.sqrt(np.mean(closure**2))))
print("Clear-sky closure max abs [W/m²]:", float(np.max(np.abs(closure))))

In [ ]:
# Exportación + metadatos de trazabilidad
cols_out = [
    "timestamp", COL_THZ,
    COL_CSGHI, COL_CSDNI, COL_CSDHI,
    "cosz", "cosz_plus"
]
physical_2050 = physical_2050[cols_out].copy()
physical_2050.to_csv(cfg.output_path, index=False)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    "artifact": cfg.output_path,
    "rows": int(len(physical_2050)),
    "columns": cols_out,
    "construction": "hybrid: pvlib zenith + historical clear-sky climatology",
    "clear_sky_reference_period": [
        str(df["timestamp"].min()),
        str(df["timestamp"].max())
    ],
    "clear_sky_statistic": "median by (doy, 30-min slot), circular rolling median 31 days",
    "site": {
        "latitude": cfg.latitude,
        "longitude": cfg.longitude,
        "altitude_m": cfg.altitude,
        "timezone": cfg.timezone,
    },
    "target_year": cfg.target_year,
    "frequency_minutes": cfg.freq_minutes,
    "sha256": sha256(cfg.output_path),
}

with open(cfg.metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Exportado:", cfg.output_path)
print("Metadata :", cfg.metadata_path)
print("Shape    :", physical_2050.shape)

## Resultado metodológico

El artefacto generado por este notebook queda congelado y será **consumido**, no recalculado, por el Notebook 3.  
Esto separa la física solar del aprendizaje neuronal y facilita la trazabilidad del artículo.